# 1. LoRA and QLoRA: a small, runnable fine-tuning demo

**Goal:** adapt a small language model using parameter-efficient fine-tuning.

This notebook chooses the safest available path automatically:

- **If an NVIDIA GPU and `bitsandbytes` are available:** it loads Qwen2.5-0.5B in 4-bit and trains LoRA adapters (QLoRA).
- **If the runtime has no compatible GPU:** it uses a very small GPT-2 model and trains LoRA on CPU. The notebook continues instead of stopping with a GPU error.

> To demonstrate actual **QLoRA**, choose **Runtime → Change runtime type → T4 GPU** before running the notebook. A CPU runtime cannot perform bitsandbytes 4-bit QLoRA. The CPU path is an honest LoRA fallback, not QLoRA.

In [1]:
import sys
import subprocess
import importlib.util
import torch


def install_if_missing(import_name, package_name):
    """Install a package only when it is not already available."""
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_name
        ])
    else:
        print(f"Already installed: {package_name}")


install_if_missing("transformers", "transformers")
install_if_missing("peft", "peft")
install_if_missing("accelerate", "accelerate")

GPU_AVAILABLE = torch.cuda.is_available()
BITSANDBYTES_AVAILABLE = False

if GPU_AVAILABLE:
    # bitsandbytes is needed only for the actual 4-bit QLoRA route.
    try:
        import bitsandbytes  # noqa: F401
        BITSANDBYTES_AVAILABLE = True
    except Exception:
        try:
            print("Installing bitsandbytes for 4-bit QLoRA...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"
            ])
            import bitsandbytes  # noqa: F401
            BITSANDBYTES_AVAILABLE = True
        except Exception as error:
            print("bitsandbytes could not be loaded. We will use the LoRA fallback.")
            print("Reason:", str(error)[:300])

print("PyTorch:", torch.__version__)
print("CUDA available:", GPU_AVAILABLE)
if GPU_AVAILABLE:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook will use a small CPU LoRA model.")

Already installed: transformers
Already installed: peft
Already installed: accelerate
PyTorch: 2.14.1+cpu
CUDA available: False
No GPU detected. The notebook will use a small CPU LoRA model.


In [10]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# Use real QLoRA only when both CUDA and bitsandbytes are available.
USE_QLORA = GPU_AVAILABLE and BITSANDBYTES_AVAILABLE
DEVICE = torch.device("cuda:0" if GPU_AVAILABLE else "cpu")

if USE_QLORA:
    print("Mode: QLoRA (4-bit base model + trainable LoRA adapters)")
else:
    print("Mode: LoRA fallback (small model, no 4-bit quantization)")

# A fixed seed makes this small experiment more repeatable.
torch.manual_seed(42)
if GPU_AVAILABLE:
    torch.cuda.manual_seed_all(42)

Mode: LoRA fallback (small model, no 4-bit quantization)


## Step 1 — Create a tiny dataset

Each row contains one instruction and one example answer. The dataset is written directly in Python, so there is no separate JSONL file to upload.

For a real fine-tuning task, use a larger, reviewed dataset and keep a separate test set that is never used for training.

In [3]:
training_examples = [
    {
        "question": "What is lift?",
        "answer": "Lift is the aerodynamic force perpendicular to the relative airflow."
    },
    {
        "question": "What is drag?",
        "answer": "Drag is the aerodynamic force parallel and opposite to the relative airflow."
    },
    {
        "question": "What is wing aspect ratio?",
        "answer": "Wing aspect ratio is the wingspan squared divided by the wing planform area."
    },
    {
        "question": "What is angle of attack?",
        "answer": "Angle of attack is the angle between the airfoil chord line and the oncoming relative airflow."
    },
    {
        "question": "What is a stall?",
        "answer": "A stall occurs when the wing exceeds its critical angle of attack and lift decreases substantially."
    },
]

print("Number of training examples:", len(training_examples))
print("First example:", training_examples[0])

Number of training examples: 5
First example: {'question': 'What is lift?', 'answer': 'Lift is the aerodynamic force perpendicular to the relative airflow.'}


## Step 2 — Load the base model

For QLoRA, we use `Qwen/Qwen2.5-0.5B-Instruct` in 4-bit NF4. If QLoRA setup fails for any reason, the notebook switches to `distilgpt2` and still demonstrates LoRA training.

In [4]:
from transformers import BitsAndBytesConfig

if USE_QLORA:
    try:
        MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # T4 GPUs use FP16 compute. The base weights are stored in 4-bit NF4.
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=quantization_config,
            device_map={"": 0},
        )

        TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
        FAN_IN_FAN_OUT = False
        print("Loaded 4-bit Qwen model successfully.")

    except Exception as error:
        # A failed 4-bit load should not kill the assignment demonstration.
        print("QLoRA model loading failed; switching to the small LoRA model.")
        print("Reason:", str(error)[:400])
        USE_QLORA = False

if not USE_QLORA:
    MODEL_NAME = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # GPT-2 tokenizers do not define a padding token by default.
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    base_model.to(DEVICE)

    TARGET_MODULES = ["c_attn"]
    FAN_IN_FAN_OUT = True
    print("Loaded the small LoRA model:", MODEL_NAME)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False
print("Final training mode:", "QLoRA" if USE_QLORA else "LoRA")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded the small LoRA model: distilgpt2
Final training mode: LoRA


## Step 3 — Format examples and test the original model

Qwen has a built-in chat template, so we use it in QLoRA mode. The small GPT-2 fallback uses a simple `Question / Answer` format.

In [5]:
SYSTEM_MESSAGE = "You are a clear and careful aerospace engineering tutor."


def format_training_example(example):
    """Convert one question-answer pair into the text expected by the model."""
    if USE_QLORA:
        messages = [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": example["answer"]},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

    return (
        f"Instruction: {example['question']}\n"
        f"Answer: {example['answer']}\n"
    )


def make_prompt(question):
    """Create an input prompt for generating an answer."""
    if USE_QLORA:
        messages = [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": question},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )

    text = f"Instruction: {question}\nAnswer:"
    return tokenizer(text, return_tensors="pt")["input_ids"]


def generate_answer(model, question, max_new_tokens=60):
    """Generate an answer using the currently loaded model."""
    input_ids = make_prompt(question).to(DEVICE)
    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output_ids[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


TEST_QUESTION = "Explain the difference between lift and drag."
original_answer = generate_answer(base_model, TEST_QUESTION)

print("MODEL BEFORE FINE-TUNING:\n")
print(original_answer)

MODEL BEFORE FINE-TUNING:

The lift is the lift that is used to lift the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight


## Step 4 — Attach LoRA adapters

LoRA freezes the original model weights and learns two smaller matrices in selected layers. In QLoRA mode, the frozen base model is also stored in 4-bit precision.

In [11]:
if USE_QLORA:
    from peft import prepare_model_for_kbit_training
    base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
    fan_in_fan_out=FAN_IN_FAN_OUT,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

W1124 17:42:18.243000 543 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


## Step 5 — Prepare the training data

We tokenize the examples and pad them to a short fixed length. Padding tokens are ignored in the loss by setting their labels to `-100`.

In [12]:
MAX_LENGTH = 160


class SimpleTextDataset(Dataset):
    """A small PyTorch dataset built directly from the examples above."""

    def __init__(self, examples):
        self.items = []

        for example in examples:
            text = format_training_example(example)
            encoded = tokenizer(
                text,
                max_length=MAX_LENGTH,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )

            input_ids = encoded["input_ids"].squeeze(0)
            attention_mask = encoded["attention_mask"].squeeze(0)
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            self.items.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


train_dataset = SimpleTextDataset(training_examples)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

print("Training examples prepared:", len(train_dataset))

Training examples prepared: 5


## Step 6 — Train the adapters

This uses a short, explicit PyTorch training loop instead of a large training framework. Only parameters with `requires_grad=True` are passed to the optimizer; those are the LoRA adapter parameters.

In [13]:
trainable_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=2e-4,
)

EPOCHS = 2
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):
        # Move this batch to the same device as the model.
        batch = {
            name: tensor.to(DEVICE)
            for name, tensor in batch.items()
        }

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        print(f"Epoch {epoch + 1}, step {step}, loss: {loss.item():.4f}")

    average_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} average loss: {average_loss:.4f}\n")

print("Training completed.")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1, step 1, loss: 4.5165
Epoch 1, step 2, loss: 4.5807
Epoch 1, step 3, loss: 4.9719
Epoch 1, step 4, loss: 4.7220
Epoch 1, step 5, loss: 4.2430
Epoch 1 average loss: 4.6068

Epoch 2, step 1, loss: 4.3675
Epoch 2, step 2, loss: 5.0638
Epoch 2, step 3, loss: 4.3458
Epoch 2, step 4, loss: 4.6116
Epoch 2, step 5, loss: 4.7847
Epoch 2 average loss: 4.6347

Training completed.


## Step 7 — Compare and save

We ask the same question after fine-tuning. A changed answer does not automatically mean the model improved; a real evaluation needs many unseen questions and human-checked expected answers.

In [14]:
model.config.use_cache = True
fine_tuned_answer = generate_answer(model, TEST_QUESTION)

print("BEFORE FINE-TUNING:\n")
print(original_answer)
print("\nAFTER FINE-TUNING:\n")
print(fine_tuned_answer)

ADAPTER_FOLDER = "./aerospace_lora_adapter"
model.save_pretrained(ADAPTER_FOLDER)
tokenizer.save_pretrained(ADAPTER_FOLDER)

print("Adapter saved to:", ADAPTER_FOLDER)
print("Training mode used:", "QLoRA" if USE_QLORA else "LoRA fallback")

BEFORE FINE-TUNING:

The lift is the lift that is used to lift the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight

AFTER FINE-TUNING:

The lift is the lift that is used to lift the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight of the weight
Adapter saved to: ./aerospace_lora_adapter
Training mode used: LoRA fallback


\## notebook demonstrates

- **LoRA:** freeze the base model and train small low-rank adapters.
- **QLoRA:** load the frozen base model in 4-bit NF4 and train LoRA adapters (only when a compatible NVIDIA GPU is available).
- **Evaluation:** compare the same prompt before and after training, without claiming that a tiny dataset proves improvement.
- **Artifact:** save the adapter separately from the base model.